# Gaussian Scene Studio — Apple SHARP public preview
Run the code cell with a Kaggle GPU enabled. Keep it running and open the complete protected URL it prints. Apple SHARP is licensed for non-commercial scientific research only.

In [ ]:
import os, re, secrets, socket, stat, subprocess, sys, time, urllib.request
from pathlib import Path

for process_name in ('server', 'tunnel'):
    previous = globals().get(process_name)
    if previous is not None and previous.poll() is None:
        previous.terminate(); previous.wait(timeout=10)
ROOT = Path('/kaggle/working/gaussian_studio')
if ROOT.exists():
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/shreyan21/gaussian_studio.git', str(ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT/'requirements.txt'), '-r', str(ROOT/'requirements-sharp.txt')], check=True)
subprocess.run([sys.executable, str(ROOT/'scripts/download_models.py'), '--model', 'sharp'], cwd=ROOT, check=True)
subprocess.run([sys.executable, str(ROOT/'scripts/download_models.py'), '--model', 'foreground'], cwd=ROOT, check=True)

cloudflared = Path('/kaggle/working/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)
token = secrets.token_urlsafe(32)
with socket.socket() as probe:
    probe.bind(('127.0.0.1', 0)); port = probe.getsockname()[1]
env = os.environ.copy(); env['GSS_ACCESS_TOKEN'] = token; env['GSS_DATA_DIR'] = f'/kaggle/working/gaussian-studio-data-{port}'
server_log = open('/kaggle/working/gaussian-studio-server.log', 'w')
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'studio.server:app', '--host', '127.0.0.1', '--port', str(port)], cwd=ROOT, env=env, stdout=server_log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{port}/api/health?token={token}', timeout=2); break
    except Exception: time.sleep(1)
else: raise RuntimeError('Studio did not start; inspect /kaggle/working/gaussian-studio-server.log')
public_url = None
for attempt in range(1, 5):
    print(f'Cloudflare tunnel attempt {attempt}/4...', flush=True)
    tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--url', f'http://127.0.0.1:{port}', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in iter(tunnel.stdout.readline, ''):
        print(line, end='')
        match = re.search(r'https://([a-z0-9-]+)\.trycloudflare\.com', line)
        if match and match.group(1) != 'api': public_url = match.group(0); break
        if 'failed to request quick Tunnel' in line: break
    if public_url: break
    tunnel.terminate(); time.sleep(3)
if not public_url: raise RuntimeError('Cloudflare tunnel failed after four attempts; rerun this cell')
print('\nOPEN THIS COMPLETE PROTECTED LINK:\n' + public_url + '/?token=' + token + '\n')
print('SHARP accepts one photograph. Keep Focus main subject enabled for dogs or products, tick the research acknowledgement, and drag within the clamped view range.')
print('Keep this cell running. Stop it to close the public link.')
while server.poll() is None and tunnel.poll() is None: time.sleep(15)
raise RuntimeError('The server or tunnel stopped; rerun this cell for a new protected URL.')